In [47]:
from __future__ import print_function, division

In [ ]:
import pandas as pd

DATA_DIR = r'D:\data'
train_df = pd.read_csv(r'D:\data\dataset.csv')
test_df = pd.read_csv(r'D:\data\experiment.csv')

train_seqs = train_df['Sequence'].tolist()
test_seqs = test_df['Sequence'].tolist()

# pick up data in training set
train_idx = [243, 39, 84, 1, 134, 217, 300, 273, 276, 185, 73, 262, 122, 5, 46, 174, 36, 158, 52, 152, 210, 133, 101, 108, 7, 83, 291, 129, 274, 189, 303, 82, 279, 59, 29, 34, 164, 252, 43, 254, 192, 136, 51, 200, 183, 227, 20, 15, 261, 23, 25, 105, 205, 302, 176, 194, 266, 203, 21, 209, 301, 223, 201, 110, 30, 155, 159, 116, 123, 222, 127, 187, 8, 67, 139, 41, 290, 79, 85, 44, 93, 71, 268, 154, 10, 90, 204, 295, 69, 141, 50, 214, 229, 118, 168, 135, 48, 115, 153, 72, 126, 172, 57, 149, 144, 9, 98, 13, 207, 211, 246, 74, 213, 267, 245, 240, 146, 182, 64, 226, 53, 173, 143, 293, 33, 75, 6, 286, 230, 269, 99, 11, 63, 91, 255, 22, 191, 68, 76, 96, 177, 94, 128, 95, 239, 150, 138, 218, 145, 250, 297, 117, 16, 280, 188, 119, 169, 249, 131, 281, 24, 219, 47, 196, 206, 113, 231, 179, 28, 80, 142, 260, 271, 170, 285, 277, 140, 65, 62, 130, 60, 107, 292, 78, 3, 208, 258, 17, 157, 106, 298, 299, 181, 42, 114, 32, 263, 27, 49, 264, 175, 242, 287, 237, 45, 221, 294, 278, 289, 100, 26, 35, 224, 120]


X_train = [train_seqs[i] for i in train_idx]

In [ ]:
import csv

def rotation_invariant_identity(seq1: str, seq2: str) -> float:
    """
    Compute the maximum pairwise identity between seq1 and any rotation of seq2.
    Uses simple position-wise identity (no gaps).
    """
    n1, n2 = len(seq1), len(seq2)
    if n1 != n2:
        min_len = min(n1, n2)
        seq1 = seq1[:min_len]
        seq2 = seq2[:min_len]
        n = min_len
    else:
        n = n1

    max_ident = 0.0
    for shift in range(n):
        rotated = seq2[shift:] + seq2[:shift]
        matches = sum(a == b for a, b in zip(seq1, rotated))
        ident = matches / n
        if ident > max_ident:
            max_ident = ident
    return max_ident

def global_identity(seq1, seq2):
    min_len = min(len(seq1), len(seq2))
    matches = sum(a == b for a, b in zip(seq1[:min_len], seq2[:min_len]))
    return matches / max(len(seq1), len(seq2))


results = []

# find maxidentity ----
for test_seq in test_seqs:
    
    best_train_seq = None
    best_identity = 0.0
    
    for train_id, train_seq in X_train:
        ident = global_identity(test_seq, train_seq)
        if ident > best_identity:
            best_identity = ident
            best_train_seq = train_seq
    
    if best_identity >= 0.6:
        best_identity = rotation_invariant_identity(test_seq, best_train_seq)
    
    results.append((test_seq, best_train_seq, best_identity))

with open('Similarity Report.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['test_seq', 'best_train_seq', 'max_identity_percent'])
    for test_seq, train_seq, ident in results:
        writer.writerow([test_seq, train_seq, f"{ident*100:.2f}%"])

idents = [r[2] for r in results]
print(f"mean identity: {sum(idents)/len(idents)*100:.2f}%")
print(f"max identity: {max(idents)*100:.2f}%")
print(f"Proportion of sequences with less than 40% identity: {sum(1 for x in idents if x<0.40)/len(idents)*100:.2f}%")

mean similarity: 34.57%
max similarity: 75.00%
Proportion of sequences with less than 40% similarity: 67.57%
